Artifact: one ML table , one row per order.
--

In [1]:
import sqlite3
import pandas as pd
import os

In [1]:
import sqlite3
import pandas as pd
import os
import sys
from pathlib import Path

# Add project src folder to Python path
PROJECT_ROOT = Path.cwd().parent                          # .parent means: Go one folder up.
sys.path.insert(0, str(PROJECT_ROOT / "src"))             # look inside my project's src folder when I use import.
                                                                # sys.path is simply a list of places Python searches.  <---
                                                                # make str(PROJECT_ROOT / "src") first place in the list to find config_loader
                                                          # .insert(0, ...): This adds the src folder to the beginning of Python's search list.
                                                          # sys.path is a list of folders where Python looks for modules to import.
                                                          # The sys.path.insert(...) line creates the connection:
                                                                #  Notebook
                                                                #    ↓
                                                                # Python search path
                                                                #    ↓
                                                                # src/
                                                                #    ↓
                                                                # config_loader.py

from config_loader import load_config

# Load project configuration
config = load_config()


1. create and Connect to database
-------

In [ ]:
# db_path = r"D:\2026\MLOps-Qafza-2026\database\olist.db"

# conn = sqlite3.connect(db_path)

db_path = PROJECT_ROOT / config["paths"]["database"]    # means create the full path to your database file.

conn = sqlite3.connect(db_path)


2. create tables and loading csv data inside
-----

In [3]:
# dataset_path = r"D:\2026\MLOps-Qafza-2026\dataset"
dataset_path = PROJECT_ROOT / config["paths"]["raw_data"]

# # Get all CSV files in the dataset folder
csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]

print("Loading CSV files into database...")
print("=" * 50)
# Loop through each CSV file , create table, read csv file and load data inside each table
for csv_file in csv_files:
    # Remove '_dataset.csv' or '.csv' from filename to use as table name
    if '_dataset.csv' in csv_file:
        table_name = csv_file.replace('_dataset.csv', '')
    else:
        table_name = csv_file.replace('.csv', '')
    
    # Full path to the CSV file
    file_path = os.path.join(dataset_path, csv_file)
    
    # Read CSV
    df = pd.read_csv(file_path)
    
    # Save to SQLite
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Loaded: '{csv_file}' to table name: {table_name} ")

Loading CSV files into database...
Loaded: 'olist_customers_dataset.csv' to table name: olist_customers 
Loaded: 'olist_geolocation_dataset.csv' to table name: olist_geolocation 
Loaded: 'olist_orders_dataset.csv' to table name: olist_orders 
Loaded: 'olist_order_items_dataset.csv' to table name: olist_order_items 
Loaded: 'olist_order_payments_dataset.csv' to table name: olist_order_payments 
Loaded: 'olist_order_reviews_dataset.csv' to table name: olist_order_reviews 
Loaded: 'olist_products_dataset.csv' to table name: olist_products 
Loaded: 'olist_sellers_dataset.csv' to table name: olist_sellers 
Loaded: 'product_category_name_translation.csv' to table name: product_category_name_translation 


In [4]:
# conform that all tables created and csv data loaded inside correctly
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn) # there will be: index and 'name' column

print(f"Total tables found: {len(tables)}")
print("=" * 75)

cursor = conn.cursor()
for table_name in tables["name"]:
    # Count rows
    cursor.execute(f'SELECT COUNT(*) FROM "{table_name}"')
    row_count = cursor.fetchone()[0]  # fetchone output as [(99441,)]

    # Count columns
    # PRAGMA is a SQLite command used to inspect or configure the database.
    # table_info is a specific SQLite PRAGMA that returns information about the columns of a table.
    cursor.execute(f'PRAGMA table_info("{table_name}")')
    columns = cursor.fetchall()
    column_count = len(columns)
    column_names = [column[1] for column in columns]
    # column[1] as PRAGMA table_info() returns information like: (column_id, column_name, data_type, not_null, default, primary_key)

    print(f"• {table_name:<35} {row_count:>10,} rows, {column_count:>3} columns having names: {column_names}")

Total tables found: 9
• olist_customers                         99,441 rows,   5 columns having names: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
• olist_geolocation                    1,000,163 rows,   5 columns having names: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
• olist_order_items                      112,650 rows,   7 columns having names: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
• olist_order_payments                   103,886 rows,   5 columns having names: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
• olist_order_reviews                     99,224 rows,   7 columns having names: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
• olist_orders      

3. understand Tables, Olist Database Relationships
------



| Table | One row represents | Primary Key | Foreign Key / Connection | Relationship |
|---|---|---|---|---|
| `olist_customers` | One customer record | `customer_id` | `customer_id` → `olist_orders_dataset.customer_id` | Customer 1 → Many Orders |
| `olist_geolocation` | One geolocation record for a ZIP-code prefix | No simple PK | `geolocation_zip_code_prefix` | Lookup / geographical reference |
| `olist_order_items` | One product item within an order | `order_id` + `order_item_id` | `order_id` → Orders<br>`product_id` → Products<br>`seller_id` → Sellers | Order 1 → Many Items |
| `olist_order_payments` | One payment record for an order | `order_id` + `payment_sequential` | `order_id` → `olist_orders_dataset.order_id` | Order 1 → Many Payments |
| `olist_order_reviews` | One review for an order | `order_id` + `review_id` | `order_id` → `olist_orders_dataset.order_id` | Order 1 → Many Reviews |
| `olist_orders` | One customer order | `order_id` | `customer_id` → `olist_customers_dataset.customer_id` | Central table |
| `olist_products` | One product | `product_id` | Connected through `order_items.product_id` | Product 1 → Many Items |
| `olist_sellers` | One seller | `seller_id` | Connected through `order_items.seller_id` | Seller 1 → Many Items |
| `product_category_name_translation` | One category translation | `product_category_name` | `product_category_name` → Products | Category 1 → Many Products |

4. check if any duplicated values in PKs
-----------

In [5]:
# check single-column PKs if having duplicated or null values
pk_checks = {
    "olist_customers": "customer_id",
    "olist_orders": "order_id",
    "olist_products": "product_id",
    "olist_sellers": "seller_id",
    "product_category_name_translation": "product_category_name",

    "olist_order_items": "order_id",
    "olist_order_payments": "order_id",
    "olist_order_reviews": "order_id"
    
}


for table, column in pk_checks.items():

    duplicate_query = f"""
    SELECT "{column}", COUNT(*) AS count
    FROM "{table}"
    GROUP BY "{column}"
    HAVING COUNT(*) > 1;
    """

    null_query = f"""
        SELECT COUNT(*) AS null_count
        FROM "{table}"
        WHERE "{column}" IS NULL;
        """

    duplicates = pd.read_sql_query(duplicate_query, conn)
    null       = pd.read_sql_query(null_query, conn)
    nulls_count = null.loc[0, "null_count"]

    print("=" * 45)
    print(f"Table : {table}")
    print(f"PK    : {column}")
    print(f"Duplicate rows found: {len(duplicates)} and null rows found: {nulls_count}")

    if duplicates.empty:
        print("✅ No duplicates")
    else:
        print("❌ Duplicates found")
        print(duplicates.head())

    # ========================
    
    if nulls_count == 0:
        print("✅ No null cells")
    else:
        print("❌ null cells found")
        

Table : olist_customers
PK    : customer_id
Duplicate rows found: 0 and null rows found: 0
✅ No duplicates
✅ No null cells
Table : olist_orders
PK    : order_id
Duplicate rows found: 0 and null rows found: 0
✅ No duplicates
✅ No null cells
Table : olist_products
PK    : product_id
Duplicate rows found: 0 and null rows found: 0
✅ No duplicates
✅ No null cells
Table : olist_sellers
PK    : seller_id
Duplicate rows found: 0 and null rows found: 0
✅ No duplicates
✅ No null cells
Table : product_category_name_translation
PK    : product_category_name
Duplicate rows found: 0 and null rows found: 0
✅ No duplicates
✅ No null cells
Table : olist_order_items
PK    : order_id
Duplicate rows found: 9803 and null rows found: 0
❌ Duplicates found
                           order_id  count
0  0008288aa423d2a3f00fcb17cd7d8719      2
1  00143d0f86d6fbd9f9b38ab440ac16f5      3
2  001ab0a7578dd66cd4b0a71f5b6e1e41      3
3  001d8f0e34a38c37f7dba2a37d4eba8b      2
4  002c9def9c9b951b1bec6d50753c9891      2

In [6]:

# Check composite-column PKs if having duplicated or null values as a composit

pk_checks = {
    "olist_order_items": ["order_id", "order_item_id"],
    "olist_order_payments": ["order_id", "payment_sequential"],
    "olist_order_reviews": ["order_id", "review_id"],
}


for table, columns in pk_checks.items():

    # Build the GROUP BY columns
    columns_sql = ", ".join(columns)

    duplicate_query = f"""
    SELECT {columns_sql}, COUNT(*) AS count
    FROM "{table}"
    GROUP BY {columns_sql}
    HAVING COUNT(*) > 1;
    """

    # Check NULL values in any column that forms the composite PK
    null_conditions = " OR ".join(
    f"{column} IS NULL" for column in columns
    )

    null_query = f"""
    SELECT COUNT(*) AS null_count
    FROM "{table}"
    WHERE {null_conditions};
    """

    duplicates  = pd.read_sql_query(duplicate_query, conn)
    null        = pd.read_sql_query(null_query, conn)
    nulls_count = null.loc[0, "null_count"]

    pk_name = " + ".join(columns)

    print("=" * 45)
    print(f"Table : {table}")
    print(f"PK    : {pk_name}")
    print(
        f"Duplicate rows found: {len(duplicates)} "
        f"and null rows found: {nulls_count}"
    )

    if duplicates.empty:
        print("✅ No duplicates")
    else:
        print("❌ Duplicates found")
        print(duplicates.head())

    # ========================

    if nulls_count == 0:
        print("✅ No null cells")
    else:
        print("❌ Null cells found")

Table : olist_order_items
PK    : order_id + order_item_id
Duplicate rows found: 0 and null rows found: 0
✅ No duplicates
✅ No null cells
Table : olist_order_payments
PK    : order_id + payment_sequential
Duplicate rows found: 0 and null rows found: 0
✅ No duplicates
✅ No null cells
Table : olist_order_reviews
PK    : order_id + review_id
Duplicate rows found: 0 and null rows found: 0
✅ No duplicates
✅ No null cells


5. Agg before join as olist_order_items, olist_order_payments, olist_order_reviews having duplicate in PK (order_id)
--------

In [7]:

# order_items has multiple rows per order.
order_items_agg_query = """
SELECT
    order_id,
    COUNT(order_item_id) AS number_of_items,
    SUM(freight_value) AS total_freight_value,
    SUM(price) AS total_price,
    COUNT(DISTINCT seller_id) AS number_of_sellers,
    COUNT(DISTINCT product_id) AS number_of_products
FROM olist_order_items
GROUP BY order_id;
"""

agg_olist_order_items = pd.read_sql_query(
    order_items_agg_query,
    conn
)

display(agg_olist_order_items)



# Save aggregated DataFrame as a new table inside db
agg_olist_order_items.to_sql(
    "agg_olist_order_items",
    conn,
    if_exists="replace",
    index=False
)


# Double-check for duplicated order_id values
duplicated_query = """
SELECT
    order_id,
    COUNT(*) AS count
FROM agg_olist_order_items
GROUP BY order_id
HAVING COUNT(*) > 1;
"""

duplicated = pd.read_sql_query(
    duplicated_query,
    conn
)

print(duplicated)

,order_id,number_of_items,total_freight_value,total_price,number_of_sellers,number_of_products
0,00010242fe8c5a6d1ba2dd792cb16214,1,13.29,58.90,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,19.93,239.90,1,1
2,000229ec398224ef6ca0657da4fc703e,1,17.87,199.00,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.79,12.99,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,18.14,199.90,1,1
...,...,...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,1,43.41,299.99,1,1
98662,fffcd46ef2263f404302a634eb57f7eb,1,36.53,350.00,1,1
98663,fffce4705a9662cd70adb13d4a31832d,1,16.95,99.90,1,1
98664,fffe18544ffabc95dfada21779c9644f,1,8.72,55.99,1,1


Empty DataFrame
Columns: [order_id, count]
Index: []


In [8]:
# order_payments has multiple rows per order.
order_payments_agg_query = """
SELECT
    order_id,
    COUNT(*) AS number_of_payments,
    SUM(payment_value) AS total_payment_value,
    COUNT(DISTINCT payment_type) AS number_of_payment_types,
    MAX(payment_installments) AS max_payment_installments
FROM olist_order_payments
GROUP BY order_id;
"""

agg_olist_order_payments = pd.read_sql_query(
    order_payments_agg_query,
    conn
)

display(agg_olist_order_payments)


# Save aggregated DataFrame as a new table inside db
agg_olist_order_payments.to_sql(
    "agg_olist_order_payments",
    conn,
    if_exists="replace",
    index=False
)


# Double-check for duplicated order_id values
duplicated_query = """
SELECT
    order_id,
    COUNT(*) AS count
FROM agg_olist_order_payments
GROUP BY order_id
HAVING COUNT(*) > 1;
"""

duplicated = pd.read_sql_query(
    duplicated_query,
    conn
)

print(duplicated)



,order_id,number_of_payments,total_payment_value,number_of_payment_types,max_payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,1,3
...,...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,1,343.40,1,1
99436,fffcd46ef2263f404302a634eb57f7eb,1,386.53,1,1
99437,fffce4705a9662cd70adb13d4a31832d,1,116.85,1,3
99438,fffe18544ffabc95dfada21779c9644f,1,64.71,1,3


Empty DataFrame
Columns: [order_id, count]
Index: []


In [9]:

# order_reviews has multiple rows per order.
order_reviews_agg_query = """
SELECT
    order_id,
    COUNT(*) AS number_of_reviews,
    AVG(review_score) AS average_review_score,
    COUNT(DISTINCT review_id) AS number_of_reviews_ids
FROM olist_order_reviews
GROUP BY order_id;
"""

agg_olist_order_reviews = pd.read_sql_query(
    order_reviews_agg_query,
    conn
)

display(agg_olist_order_reviews)


# Save aggregated DataFrame as a new table inside db
agg_olist_order_reviews.to_sql(
    "agg_olist_order_reviews",
    conn,
    if_exists="replace",
    index=False
)


# Double-check for duplicated order_id values
duplicated_query = """
SELECT
    order_id,
    COUNT(*) AS count
FROM agg_olist_order_reviews
GROUP BY order_id
HAVING COUNT(*) > 1;
"""

duplicated = pd.read_sql_query(
    duplicated_query,
    conn
)

print(duplicated)

,order_id,number_of_reviews,average_review_score,number_of_reviews_ids
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,1,4.0,1
2,000229ec398224ef6ca0657da4fc703e,1,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,1,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0,1
...,...,...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,1,5.0,1
98669,fffcd46ef2263f404302a634eb57f7eb,1,5.0,1
98670,fffce4705a9662cd70adb13d4a31832d,1,5.0,1
98671,fffe18544ffabc95dfada21779c9644f,1,5.0,1


Empty DataFrame
Columns: [order_id, count]
Index: []


6. table joining, 
-------------



now we have

| Table | Granularity |
|---|---|
| `olist_orders` | → 1 row/order |
| `agg_olist_order_items` | → 1 row/order |
| `agg_olist_order_payments` | → 1 row/order |
| `agg_olist_order_reviews` | → 1 row/order |

In [ ]:
ml_orders_query ='''
CREATE TABLE ml_orders AS
SELECT
    o.*,
    i.number_of_items,
    i.total_freight_value,
    i.total_price,
    i.number_of_sellers,
    i.number_of_products,
    p.number_of_payments,
    p.total_payment_value,
    p.number_of_payment_types,
    p.max_payment_installments
FROM olist_orders AS o
LEFT JOIN agg_olist_order_items AS i
    ON o.order_id = i.order_id
LEFT JOIN agg_olist_order_payments AS p
    ON o.order_id = p.order_id;
    '''

cursor = conn.cursor()
cursor.execute(ml_orders_query)
conn.commit()    # commit() → permanently save the change

print("✅ ml_orders table created successfully")

✅ ml_orders table created successfully


In [ ]:
# confirm that final ml_orders table created and display the data sample
query = """
SELECT *
FROM ml_orders
LIMIT 10;
"""

ml_orders = pd.read_sql_query(query, conn)

display(ml_orders)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,number_of_items,total_freight_value,total_price,number_of_sellers,number_of_products,number_of_payments,total_payment_value,number_of_payment_types,max_payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,8.72,29.99,1,1,3,38.71,2,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,22.76,118.70,1,1,1,141.46,1,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,19.22,159.90,1,1,1,179.12,1,3
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,27.20,45.00,1,1,1,72.20,1,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,8.72,19.90,1,1,1,28.62,1,1
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00,1,27.36,147.90,1,1,1,175.26,1,6
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00,1,16.05,49.90,1,1,1,65.95,1,1
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00,1,15.17,59.99,1,1,1,75.16,1,3
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00,1,16.05,19.90,1,1,1,35.95,1,1
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,1,19.77,149.99,1,1,2,169.76,2,1


In [12]:
# count the number of rows inside ml_orders table
count_query = """
SELECT COUNT(*) AS number_of_rows
FROM ml_orders;
"""

count = pd.read_sql_query(count_query, conn)

display(count)

,number_of_rows
0,99441


In [13]:
# test: ml_orders["order_id"].nunique() == len(ml_orders)
test_query = '''
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_order_ids,
    COUNT(*) = COUNT(DISTINCT order_id) AS order_id_is_unique
FROM ml_orders;
'''
test = pd.read_sql_query(test_query, conn)

display(test)


,total_rows,unique_order_ids,order_id_is_unique
0,99441,99441,1


In [ ]:
# display the names for all tables inside the db
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn) 
tables

,name
0,agg_olist_order_items
1,agg_olist_order_payments
2,agg_olist_order_reviews
3,ml_orders
4,olist_customers
5,olist_geolocation
6,olist_order_items
7,olist_order_payments
8,olist_order_reviews
9,olist_orders
